<a href="https://colab.research.google.com/github/anacvetkoo/MOS-projekt/blob/main/notebooks/Ana_teden5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

natančna specifikacija CIP11 ("Computer and Information Sciences and Support Services"):
- Splošne računalniške in informacijske vede, umetna inteligenca, informatika, IT
- Programiranje, razvoj programske opreme in specifičnih aplikacij
- Obdelava podatkov in tehnologije za obdelavo podatkov
- Informacijske vede in upravljanje informacij
- Spletne aplikacije, podatkovne baze, računalniška grafika, simulacije
- Sistemska administracija, informacijska varnost, upravljanje IT
- Računalniška omrežja, omrežni sistemi in telekomunikacije
- ...

Elektrotehnika NE spada sem


poglobljeno RV4 - struktura diplom po študijskih področjih na ivy league schools

In [41]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content")

rv4 = pd.read_csv(DATA_DIR / "IPEDS_RV4_nova.csv")

print("Obdobje:", rv4["leto"].min(), "–", rv4["leto"].max())


Obdobje: 2004 – 2024


In [42]:
#izbor ivy league in priprava diplom

ivy_ids = {
    217156: "Brown",
    190150: "Columbia",
    190415: "Cornell",
    182670: "Dartmouth",
    166027: "Harvard",
    215062: "Penn",
    186131: "Princeton",
    130794: "Yale"
}

#diplome prve stopnje, glavna študijska smer
ivy_rv4 = rv4[
    rv4["UNITID"].isin(ivy_ids) &
    (rv4["AWLEVEL"] == 5) &
    (rv4["MAJORNUM"] == 1)
].copy()

ivy_rv4["univerza"] = ivy_rv4["UNITID"].map(ivy_ids)

#poenotenje števila diplom med leti
ivy_rv4["diplome"] = ivy_rv4["CRACE24"].fillna(ivy_rv4["CTOTALT"])

print("Univerze:", ivy_rv4["univerza"].nunique())
print("Obdobje:", ivy_rv4["leto"].min(), "–", ivy_rv4["leto"].max())


Univerze: 8
Obdobje: 2004 – 2024


In [43]:
#preverjanje

#cip 99 predstavlja skupno število diplom
skupaj = (
    ivy_rv4[ivy_rv4["CIPCODE"] == 99]
    .groupby(["univerza", "leto"])["diplome"]
    .sum()
    .rename("diplome_skupaj")
)

programi_rv4 = ivy_rv4[ivy_rv4["CIPCODE"] != 99].copy()

vsota_programov = (
    programi_rv4.groupby(["univerza", "leto"])["diplome"]
    .sum()
)

#preverjanje skladnosti skupnega števila in vsote programov
assert len(skupaj) == 168
assert skupaj.equals(vsota_programov.rename("diplome_skupaj")), \
    "Neskladje med skupnim številom diplom in vsoto programov."

print("Preverjanje uspešno: vseh 168 kombinacij je skladnih.")


Preverjanje uspešno: vseh 168 kombinacij je skladnih.


In [44]:
#izracun strukture

programi_rv4["cip2"] = np.floor(programi_rv4["CIPCODE"]).astype(int)

#operativni definiciji skupin
tehnicni_cip = [11, 14, 15]
stem_cip = [11, 14, 15, 26, 27, 40]

programi_rv4["cip11_diplome"] = np.where(
    programi_rv4["cip2"] == 11, programi_rv4["diplome"], 0
)
programi_rv4["tehnicne_diplome"] = np.where(
    programi_rv4["cip2"].isin(tehnicni_cip), programi_rv4["diplome"], 0
)
programi_rv4["stem_diplome"] = np.where(
    programi_rv4["cip2"].isin(stem_cip), programi_rv4["diplome"], 0
)

agregat_rv4 = (
    programi_rv4.groupby(["univerza", "leto"])[
        ["cip11_diplome", "tehnicne_diplome", "stem_diplome"]
    ].sum()
)

struktura_rv4 = skupaj.to_frame().join(agregat_rv4).reset_index()

#izračun deležev diplom
for stolpec in ["cip11_diplome", "tehnicne_diplome", "stem_diplome"]:
    struktura_rv4["delez_" + stolpec] = (
        100 * struktura_rv4[stolpec] / struktura_rv4["diplome_skupaj"]
    )

struktura_rv4["cip11_med_tehnicnimi"] = (
    100 * struktura_rv4["cip11_diplome"] /
    struktura_rv4["tehnicne_diplome"].replace(0, np.nan)
)

display(struktura_rv4[struktura_rv4["leto"] == 2024].round(2))


,univerza,leto,diplome_skupaj,cip11_diplome,tehnicne_diplome,stem_diplome,delez_cip11_diplome,delez_tehnicne_diplome,delez_stem_diplome,cip11_med_tehnicnimi
20,Brown,2024,1944.0,282.0,386.0,888.0,14.51,19.86,45.68,73.06
41,Columbia,2024,2433.0,419.0,721.0,1121.0,17.22,29.63,46.07,58.11
62,Cornell,2024,3875.0,693.0,1204.0,1762.0,17.88,31.07,45.47,57.56
83,Dartmouth,2024,1257.0,103.0,317.0,512.0,8.19,25.22,40.73,32.49
104,Harvard,2024,1912.0,184.0,280.0,857.0,9.62,14.64,44.82,65.71
125,Penn,2024,2827.0,240.0,462.0,982.0,8.49,16.34,34.74,51.95
146,Princeton,2024,1324.0,213.0,441.0,666.0,16.09,33.31,50.30,48.30
167,Yale,2024,1665.0,161.0,270.0,640.0,9.67,16.22,38.44,59.63


metodologija: Upoštevane so diplome prve stopnje (bachelor's degrees, AWLEVEL = 5) in glavne študijske smeri (MAJORNUM = 1)

skupine za primerjavo:
- cip11
- ožja tehnična skupina (cip11,14,15) - računalniške in informacijske vede, inženirstvo ter inženirske tehnologije
- širša skupina STEM (cip11,14,15,26,27,40) - vključuje še biološke in biomedicinske vede, matematiko in statistiko ter fizikalne vede

In [45]:

#število diplom po glavnih področjih CIP
cip_podrocja = (
    programi_rv4.groupby(["univerza", "leto", "cip2"], as_index=False)
    ["diplome"].sum()
)

cip_podrocja = cip_podrocja.merge(
    struktura_rv4[["univerza", "leto", "diplome_skupaj"]],
    on=["univerza", "leto"],
    how="left",
    validate="many_to_one"
)

cip_podrocja["delez"] = (
    100 * cip_podrocja["diplome"] /
    cip_podrocja["diplome_skupaj"]
)

#preverjanje, ali se deleži vseh področij seštejejo v 100 %
preverjanje_delezev = (
    cip_podrocja.groupby(["univerza", "leto"])["delez"]
    .sum()
)

assert np.allclose(preverjanje_delezev, 100), \
    "Deleži področij se ne seštejejo v 100 %."

print("Preverjanje uspešno: deleži vseh področij se seštejejo v 100 %.")
print("Število področij CIP:", cip_podrocja["cip2"].nunique())
print("Obdobje:", cip_podrocja["leto"].min(), "–", cip_podrocja["leto"].max())

display(
    cip_podrocja[
        (cip_podrocja["univerza"] == "Brown") &
        (cip_podrocja["leto"] == 2024)
    ].sort_values("delez", ascending=False).head(15).round(2)
)


Preverjanje uspešno: deleži vseh področij se seštejejo v 100 %.
Število področij CIP: 29
Obdobje: 2004 – 2024


,univerza,leto,cip2,diplome,diplome_skupaj,delez
426,Brown,2024,45,454.0,1944.0,23.35
414,Brown,2024,11,282.0,1944.0,14.51
419,Brown,2024,26,226.0,1944.0,11.63
420,Brown,2024,27,210.0,1944.0,10.80
421,Brown,2024,30,112.0,1944.0,5.76
416,Brown,2024,14,104.0,1944.0,5.35
418,Brown,2024,23,87.0,1944.0,4.48
423,Brown,2024,40,66.0,1944.0,3.40
424,Brown,2024,42,66.0,1944.0,3.40
427,Brown,2024,50,63.0,1944.0,3.24


In [46]:

#razvrstitev področij CIP v skupine za vizualizacijo
skupine_cip = {
    11: "Računalništvo in IT",
    14: "Inženirstvo in tehnologija",
    15: "Inženirstvo in tehnologija",
    26: "Naravoslovje in matematika",
    27: "Naravoslovje in matematika",
    40: "Naravoslovje in matematika",
    45: "Družboslovje",
    52: "Poslovne vede",
    16: "Humanistika in jeziki",
    23: "Humanistika in jeziki",
    24: "Humanistika in jeziki",
    38: "Humanistika in jeziki",
    54: "Humanistika in jeziki",
    30: "Interdisciplinarni študiji"
}

cip_podrocja["skupina"] = (
    cip_podrocja["cip2"]
    .map(skupine_cip)
    .fillna("Druga področja")
)

struktura_skupine = (
    cip_podrocja.groupby(
        ["univerza", "leto", "skupina"], as_index=False
    )[["diplome", "delez"]].sum()
)

#preverjanje, ali skupine skupaj predstavljajo 100 %
vsote = (
    struktura_skupine.groupby(["univerza", "leto"])["delez"]
    .sum()
)

assert np.allclose(vsote, 100)

print("Število skupin:", struktura_skupine["skupina"].nunique())
print("Preverjanje uspešno: deleži skupin se seštejejo v 100 %.")

display(
    struktura_skupine[
        (struktura_skupine["univerza"] == "Brown") &
        (struktura_skupine["leto"] == 2024)
    ].sort_values("delez", ascending=False).round(2)
)


Število skupin: 8
Preverjanje uspešno: deleži skupin se seštejejo v 100 %.


,univerza,leto,skupina,diplome,delez
165,Brown,2024,Naravoslovje in matematika,502.0,25.82
161,Brown,2024,Družboslovje,454.0,23.35
160,Brown,2024,Druga področja,292.0,15.02
167,Brown,2024,Računalništvo in IT,282.0,14.51
162,Brown,2024,Humanistika in jeziki,193.0,9.93
163,Brown,2024,Interdisciplinarni študiji,112.0,5.76
164,Brown,2024,Inženirstvo in tehnologija,104.0,5.35
166,Brown,2024,Poslovne vede,5.0,0.26


In [47]:

import plotly.graph_objects as go

#vrstni red univerz in skupin
univerze = list(ivy_ids.values())

vrstni_red = [
    "Računalništvo in IT",
    "Inženirstvo in tehnologija",
    "Naravoslovje in matematika",
    "Družboslovje",
    "Poslovne vede",
    "Humanistika in jeziki",
    "Interdisciplinarni študiji",
    "Druga področja"
]

barve = {
    "Računalništvo in IT": "#2563eb",
    "Inženirstvo in tehnologija": "#60a5fa",
    "Naravoslovje in matematika": "#16a34a",
    "Družboslovje": "#f59e0b",
    "Poslovne vede": "#ef4444",
    "Humanistika in jeziki": "#a855f7",
    "Interdisciplinarni študiji": "#14b8a6",
    "Druga področja": "#94a3b8"
}

#priprava podatkov za posamezno leto
def podatki_leta(leto):
    podatki = struktura_skupine[
        struktura_skupine["leto"] == leto
    ]

    tabela = podatki.pivot(
        index="univerza",
        columns="skupina",
        values=["delez", "diplome"]
    ).reindex(index=univerze)

    return tabela

#ustvarjanje stolpcev za izhodiščno leto
fig = go.Figure()
zacetno_leto = 2004
tabela = podatki_leta(zacetno_leto)

for skupina in vrstni_red:
    delez = tabela["delez"].get(
        skupina, pd.Series(0, index=univerze)
    ).fillna(0)

    diplome = tabela["diplome"].get(
        skupina, pd.Series(0, index=univerze)
    ).fillna(0)

    fig.add_trace(go.Bar(
        x=univerze,
        y=delez,
        name=skupina,
        marker_color=barve[skupina],
        customdata=diplome.to_numpy(),
        hovertemplate=(
            "<b>%{x}</b><br>"
            + skupina + "<br>"
            "Delež: %{y:.2f} %<br>"
            "Število diplom: %{customdata:,.0f}"
            "<extra></extra>"
        )
    ))

#priprava posodobitev za drsnik
koraki = []

for leto in range(2004, 2025):
    tabela = podatki_leta(leto)

    novi_delezi = []
    nove_diplome = []

    for skupina in vrstni_red:
        delez = tabela["delez"].get(
            skupina, pd.Series(0, index=univerze)
        ).fillna(0)

        diplome = tabela["diplome"].get(
            skupina, pd.Series(0, index=univerze)
        ).fillna(0)

        novi_delezi.append(delez.tolist())
        nove_diplome.append(diplome.tolist())

    koraki.append(dict(
        method="update",
        args=[
            {
                "y": novi_delezi,
                "customdata": nove_diplome
            },
            {
                "title": f"Struktura diplom Ivy League – {leto}"
            }
        ],
        label=str(leto)
    ))

fig.update_layout(
    title=f"Struktura diplom Ivy League – {zacetno_leto}",
    barmode="stack",
    xaxis_title="Univerza",
    yaxis_title="Delež vseh diplom (%)",
    yaxis=dict(range=[0, 100], ticksuffix="%"),
    height=680,
    legend=dict(
        orientation="h",
        y=-0.28,
        x=0
    ),
    sliders=[dict(
        active=0,
        currentvalue=dict(prefix="Leto: "),
        pad=dict(t=55),
        steps=koraki
    )],
    margin=dict(t=85, b=175),
    hovermode="closest"
)

fig.show()


- največje zmanjšanje:humanistika in jeziki in druga področja
- vidim povečanje tehniških študijev -> prav tako se povečuje delež računalništva in IT med tehniškimi študiji

In [48]:
#podrobnejša analiza, iskanje anomalij

#letne spremembe računalništva po univerzah
trend_cip11 = struktura_rv4[
    ["univerza", "leto", "cip11_diplome",
     "diplome_skupaj", "delez_cip11_diplome"]
].copy().sort_values(["univerza", "leto"])

trend_cip11["sprememba_ot"] = (
    trend_cip11.groupby("univerza")["delez_cip11_diplome"].diff()
)

trend_cip11["sprememba_diplom"] = (
    trend_cip11.groupby("univerza")["cip11_diplome"].diff()
)

#skupni uteženi trend Ivy League
ivy_trend = (
    trend_cip11.groupby("leto", as_index=False)
    [["cip11_diplome", "diplome_skupaj"]].sum()
)

ivy_trend["delez_cip11"] = (
    100 * ivy_trend["cip11_diplome"] /
    ivy_trend["diplome_skupaj"]
)

ivy_trend["sprememba_ot"] = ivy_trend["delez_cip11"].diff()

print("Največji medletni skoki po univerzah:")
display(
    trend_cip11.dropna(subset=["sprememba_ot"])
    .sort_values("sprememba_ot", ascending=False)
    .groupby("univerza")
    .head(3)
    [["univerza", "leto", "sprememba_ot",
      "sprememba_diplom", "delez_cip11_diplome"]]
    .sort_values(["univerza", "sprememba_ot"],
                 ascending=[True, False])
    .round(2)
)

print("\nNajvečji skoki Ivy League kot celote:")
display(
    ivy_trend.nlargest(5, "sprememba_ot")
    [["leto", "delez_cip11", "sprememba_ot"]]
    .round(2)
)


Največji medletni skoki po univerzah:


,univerza,leto,sprememba_ot,sprememba_diplom,delez_cip11_diplome
13,Brown,2017,2.00,29.0,9.48
17,Brown,2021,1.80,6.0,14.08
12,Brown,2016,1.57,22.0,7.48
34,Columbia,2017,2.77,43.0,8.98
35,Columbia,2018,2.34,52.0,11.32
41,Columbia,2024,2.11,54.0,17.22
59,Cornell,2021,2.54,93.0,15.35
54,Cornell,2016,2.29,91.0,8.09
60,Cornell,2022,1.65,52.0,17.01
81,Dartmouth,2022,2.86,21.0,11.90



Največji skoki Ivy League kot celote:


,leto,delez_cip11,sprememba_ot
15,2019,9.58,1.55
16,2020,10.80,1.22
12,2016,5.87,1.20
13,2017,7.05,1.19
17,2021,11.93,1.13


opazim anomalijo - Princeton največji skok. delež diplom iz cip11 šel iz 0% na 11,26% (146) -> grem to preverit. možno da gre za pomanjkljive podatke

In [49]:

#princeton: spremembe števila diplom po vseh področjih CIP
p = cip_podrocja[
    (cip_podrocja["univerza"] == "Princeton") &
    (cip_podrocja["leto"].between(2016, 2020))
]

tabela = p.pivot_table(
    index="cip2",
    columns="leto",
    values="diplome",
    aggfunc="sum",
    fill_value=0
)

tabela["sprememba_2018_2019"] = tabela[2019] - tabela[2018]

display(
    tabela.sort_values("sprememba_2018_2019", ascending=False)
)


leto,2016,2017,2018,2019,2020,sprememba_2018_2019
cip2,,,,,,
11,0.0,0.0,0.0,146.0,158.0,146.0
45,246.0,259.0,237.0,274.0,253.0,37.0
40,80.0,77.0,55.0,73.0,86.0,18.0
38,35.0,48.0,34.0,41.0,41.0,7.0
44,141.0,107.0,116.0,122.0,108.0,6.0
26,123.0,117.0,135.0,141.0,127.0,6.0
42,68.0,45.0,33.0,36.0,40.0,3.0
27,29.0,32.0,36.0,35.0,37.0,-1.0
4,10.0,9.0,10.0,8.0,16.0,-2.0


računalništvo (cip11) se povečalo za 146, inženirstvo (cip14) pa se zmanjšalo za 164 -> spremembe razvrščanja diplom med cip11 in cip14?

In [50]:

#podrobne kode inženirstva na Princetonu
princeton_inzenirstvo = programi_rv4[
    (programi_rv4["univerza"] == "Princeton") &
    (programi_rv4["leto"].between(2016, 2020)) &
    (programi_rv4["cip2"] == 14)
]

tabela_cip14 = princeton_inzenirstvo.pivot_table(
    index="CIPCODE",
    columns="leto",
    values="diplome",
    aggfunc="sum",
    fill_value=0
)

tabela_cip14["sprememba_2018_2019"] = (
    tabela_cip14[2019] - tabela_cip14[2018]
)

display(
    tabela_cip14.sort_values(
        "sprememba_2018_2019"
    )
)


leto,2016,2017,2018,2019,2020,sprememba_2018_2019
CIPCODE,,,,,,
14.0901,112.0,129.0,167.0,0.0,0.0,-167.0
14.0701,45.0,38.0,49.0,39.0,26.0,-10.0
14.1901,44.0,39.0,50.0,48.0,52.0,-2.0
14.0101,0.0,0.0,0.0,0.0,0.0,0.0
14.1001,35.0,33.0,43.0,44.0,21.0,1.0
14.0801,25.0,29.0,17.0,21.0,17.0,4.0
14.3701,77.0,72.0,56.0,66.0,69.0,10.0


anomalija rešena! ni prišlo do takšnega zvišanja cip11 iz 2018 v 2019, ampak so te programe uvrščali pod cip 14.
- CIP 14.0901 – Computer Engineering
- CIP 11.0701 – Computer Science

In [51]:

#primerjava računalništva in računalniškega inženirstva na Princetonu
princeton_kodi = programi_rv4[
    (programi_rv4["univerza"] == "Princeton") &
    (programi_rv4["CIPCODE"].isin([11.0701, 14.0901]))
]

primerjava_kod = (
    princeton_kodi.pivot_table(
        index="leto",
        columns="CIPCODE",
        values="diplome",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(range(2004, 2025), fill_value=0)
)

primerjava_kod["skupaj"] = primerjava_kod.sum(axis=1)

display(primerjava_kod)


CIPCODE,11.0701,14.0901,skupaj
leto,,,
2004,0.0,37.0,37.0
2005,0.0,37.0,37.0
2006,0.0,28.0,28.0
2007,0.0,18.0,18.0
2008,0.0,41.0,41.0
2009,0.0,29.0,29.0
2010,0.0,44.0,44.0
2011,0.0,36.0,36.0
2012,0.0,52.0,52.0


nadaljujem zdaj s pregledovanjem podatkov, iskanjem anomalij itd.

In [52]:

#celotna dinamika prilagojenega trenda Ivy League
pregled_rasti = ivy_obcutljivost[
    ["leto", "delez_cip11", "delez_prilagojen",
     "sprememba_prilagojena_ot"]
].copy()

#triletna sprememba deleža v odstotnih točkah
pregled_rasti["sprememba_3leta_ot"] = (
    pregled_rasti["delez_prilagojen"] -
    pregled_rasti["delez_prilagojen"].shift(3)
)

print("Celoten pregled rasti:")
display(pregled_rasti.round(2))

print("\nNajvečja triletna povečanja:")
display(
    pregled_rasti.nlargest(5, "sprememba_3leta_ot")
    [["leto", "sprememba_3leta_ot"]]
    .round(2)
)

print("\nLeta z zmanjšanjem prilagojenega deleža:")
display(
    pregled_rasti[
        pregled_rasti["sprememba_prilagojena_ot"] < 0
    ][["leto", "sprememba_prilagojena_ot"]]
    .round(2)
)


Celoten pregled rasti:


,leto,delez_cip11,delez_prilagojen,sprememba_prilagojena_ot,sprememba_3leta_ot
0,2004,2.83,3.08,NaN,NaN
1,2005,2.16,2.41,-0.67,NaN
2,2006,1.53,1.72,-0.69,NaN
3,2007,1.59,1.71,-0.01,-1.37
4,2008,1.48,1.76,0.05,-0.65
5,2009,1.51,1.71,-0.05,-0.01
6,2010,1.83,2.12,0.41,0.41
7,2011,2.17,2.40,0.29,0.65
8,2012,2.35,2.69,0.29,0.98
9,2013,2.88,3.34,0.65,1.22



Največja triletna povečanja:


,leto,sprememba_3leta_ot
14,2018,3.68
13,2017,3.55
18,2022,3.24
12,2016,3.23
15,2019,3.02



Leta z zmanjšanjem prilagojenega deleža:


,leto,sprememba_prilagojena_ot
1,2005,-0.67
2,2006,-0.69
3,2007,-0.01
5,2009,-0.05
19,2023,-0.27


In [53]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots

#skupni deleži za Ivy League po letih
rv4_skupaj = (
    struktura_rv4.groupby("leto", as_index=False)[
        ["diplome_skupaj", "cip11_diplome", "tehnicne_diplome"]
    ].sum()
)

rv4_skupaj["delez_tehnicnih"] = (
    100 * rv4_skupaj["tehnicne_diplome"] /
    rv4_skupaj["diplome_skupaj"]
)

rv4_skupaj["cip11_med_tehnicnimi"] = (
    100 * rv4_skupaj["cip11_diplome"] /
    rv4_skupaj["tehnicne_diplome"]
)

#alternativni delež CIP 11 med tehniškimi diplomami
rv4_skupaj["princeton_140901"] = (
    rv4_skupaj["leto"].map(princeton_popravek).fillna(0)
)

rv4_skupaj["cip11_med_tehnicnimi_prilagojen"] = (
    100 * (
        rv4_skupaj["cip11_diplome"] +
        rv4_skupaj["princeton_140901"]
    ) / rv4_skupaj["tehnicne_diplome"]
)

rv4_skupaj = rv4_skupaj.merge(
    ivy_obcutljivost[
        ["leto", "delez_cip11", "delez_prilagojen"]
    ],
    on="leto",
    validate="one_to_one"
)

#dva povezana časovna prikaza
fig_rv4 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.13,
    subplot_titles=(
        "Delež med vsemi diplomami",
        "Delež računalništva med tehniškimi diplomami"
    )
)

def dodaj_crto(stolpec, ime, barva, vrstica, crtkano=False):
    fig_rv4.add_trace(
        go.Scatter(
            x=rv4_skupaj["leto"],
            y=rv4_skupaj[stolpec],
            mode="lines+markers",
            name=ime,
            line=dict(
                color=barva,
                width=2 if crtkano else 3,
                dash="dash" if crtkano else "solid"
            ),
            marker=dict(size=5),
            hovertemplate=(
                "Leto: %{x}<br>"
                "Delež: %{y:.2f} %<extra>%{fullData.name}</extra>"
            )
        ),
        row=vrstica, col=1
    )

#zgornji graf
dodaj_crto("delez_tehnicnih", "Tehniške diplome (CIP 11, 14, 15)", "#16a34a", 1)
dodaj_crto("delez_cip11", "Računalništvo – uradni CIP 11", "#94a3b8", 1, True)
dodaj_crto("delez_prilagojen", "Računalništvo – prilagojeno", "#2563eb", 1)

#spodnji graf
dodaj_crto("cip11_med_tehnicnimi", "CIP 11 med tehniškimi – uradno", "#94a3b8", 2, True)
dodaj_crto("cip11_med_tehnicnimi_prilagojen", "CIP 11 med tehniškimi – prilagojeno", "#7c3aed", 2)

#označitev obdobja izrazite rasti
for vrstica in [1, 2]:
    fig_rv4.add_vrect(
        x0=2013, x1=2018,
        fillcolor="#bfdbfe",
        opacity=0.18,
        line_width=0,
        row=vrstica, col=1
    )
    fig_rv4.add_vline(
        x=2019,
        line=dict(color="#dc2626", dash="dot", width=1.5),
        row=vrstica, col=1
    )

fig_rv4.update_layout(
    title=dict(
        text="Računalništvo in tehniška področja na Ivy League (2004–2024)",
        x=0.5
    ),
    template="plotly_white",
    height=800,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        x=0,
        y=-0.13
    ),
    margin=dict(t=110, b=150, l=75, r=35)
)

fig_rv4.update_xaxes(
    title_text="Leto",
    tickmode="linear",
    dtick=2,
    range=[2004, 2024],
    row=2, col=1
)

fig_rv4.update_yaxes(
    title_text="Delež vseh diplom (%)",
    ticksuffix="%",
    rangemode="tozero",
    row=1, col=1
)

fig_rv4.update_yaxes(
    title_text="Delež tehniških diplom (%)",
    ticksuffix="%",
    range=[0, 100],
    row=2, col=1
)

fig_rv4.show()


UGOTOVITVE:
- delež tehniških področij se je povečal s približno 13 % leta 2004 na 24 % leta 2024
- Računalništvo je raslo še izraziteje: delež med vsemi diplomami se je povečal s 3,08 % na 13,31 %
- Računalništvo postaja prevladujoče znotraj opredeljenih tehniških področij: njegov prilagojeni delež med tehniškimi diplomami se je povečal s približno 24 % na 56 %. Od leta 2020 predstavlja več kot polovico teh diplom.
- Rast ni bila povsem enakomerna: po izrazitem pospeševanju med letoma 2013 in 2018 se je nadaljevala tudi po letu 2019, z manjšim upadom deleža računalništva leta 2023.

To pomeni, da rast tehniških področij ni bila enakomerno razporejena med računalništvo in inženirstvo, ampak je računalništvo znotraj tehniške skupine pridobivalo relativni pomen.

upad deleža tehniških in cip11 diplom iz 2022 v 2023?? -> grem raziskat

In [54]:

#preverjanje upada deleža računalništva leta 2023
primerjava_2023 = rv4_skupaj[
    rv4_skupaj["leto"].isin([2022, 2023, 2024])
][["leto", "cip11_diplome", "diplome_skupaj", "delez_prilagojen"]].copy()

primerjava_2023["ostale_diplome"] = (
    primerjava_2023["diplome_skupaj"] -
    primerjava_2023["cip11_diplome"]
)

display(primerjava_2023.round(2))


,leto,cip11_diplome,diplome_skupaj,delez_prilagojen,ostale_diplome
18,2022,2046.0,15960.0,12.82,13914.0
19,2023,2071.0,16503.0,12.55,14432.0
20,2024,2295.0,17237.0,13.31,14942.0


ugotovitev: število računalniških diplom se je iz 2022 na 2023 povečalo (za 25) ampak se je skupno število diplom povečalo še bolj - zato upad deleža računalništva

Število, delež računalniških diplom na Ivy league v primerjavi z nacionalnim trendom - vključitev že obstoječe analize
- Naslov: Undergraduate Degree Fields
- Avtor: National Center for Education Statistics (NCES), ameriško ministrstvo za izobraževanje
- Vir: IPEDS Completions
- Obdobje: 2011/12 - 2021/22

Raziskava ugotavlja, da se je v ZDA število diplom prve stopnje s področja računalništva in informacijskih ved povečalo s približno 47.400 na 108.500, kar predstavlja 129-odstotno rast.

In [55]:
#primerjava z NCES: uradni CIP 11 in prilagojeni kazalnik
p = rv4_skupaj.set_index("leto")

for stolpec, naziv in [
    ("cip11_diplome", "Uradni CIP 11"),
    ("cip11_prilagojeno", "Prilagojeni CIP 11")
]:
    if stolpec == "cip11_prilagojeno":
        zacetek = p.loc[2012, "cip11_diplome"] + p.loc[2012, "princeton_140901"]
        konec = p.loc[2022, "cip11_diplome"] + p.loc[2022, "princeton_140901"]
    else:
        zacetek = p.loc[2012, stolpec]
        konec = p.loc[2022, stolpec]

    rast = (konec / zacetek - 1) * 100
    print(f"{naziv}: {zacetek:.0f} → {konec:.0f} diplom ({rast:+.1f} %)")

print("NCES – ZDA, uradni CIP 11: +129 %")

print("\nPreverjanje oznak izvornih datotek:")
display(
    rv4[rv4["leto"].isin([2012, 2022])]
    .groupby("leto")["vir"].unique()
)


Uradni CIP 11: 364 → 2046 diplom (+462.1 %)
Prilagojeni CIP 11: 416 → 2046 diplom (+391.8 %)
NCES – ZDA, uradni CIP 11: +129 %

Preverjanje oznak izvornih datotek:


,vir
leto,
2012,[C2012_A]
2022,[C2022_A]


Na Ivy League se je število računalniških diplom povečalo precej hitreje kot na ravni ZDA.

Rezultati: NCES ugotavlja 129-odstotno rast računalniških diplom v ZDA. Na univerzah Ivy League sem v istem obdobju ugotovila 462,1-odstotno rast uradnega kazalnika CIP 11 (364 → 2.046 diplom). Zaradi ugotovljenega preloma evidentiranja diplom na Princetonu sem izvedla še analizo občutljivosti, pri kateri rast znaša 391,8 % (416 → 2.046 diplom).

Ugotovitev: Obe analizi kažeta izrazito rast računalništva, vendar je bila relativna rast na Ivy League bistveno večja kot na nacionalni ravni. Ugotovitev ostane enaka tudi po prilagoditvi za Princeton.

Poglobljen map - primerjava east coast proti west coast

In [57]:

import pandas as pd
import numpy as np
import json
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from IPython.display import HTML, display

#poti in datoteke
DATA_DIR = Path("/content")
FIG_DIR = DATA_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

rv1 = pd.read_csv(DATA_DIR / "IPEDS_RV1.csv", low_memory=False)
rv2 = pd.read_csv(DATA_DIR / "IPEDS_RV2.csv", low_memory=False)
rv4 = pd.read_csv(DATA_DIR / "IPEDS_RV4_nova.csv", low_memory=False)
inst = pd.read_csv(DATA_DIR / "IPEDS_institucije_2004_2024.csv")

#koordinate iz IPEDS
url = "https://nces.ed.gov/ipeds/datacenter/data/HD2024.zip"
hd = pd.read_csv(
    url, compression="zip", encoding="utf-8-sig",
    low_memory=False
)
hd.columns = hd.columns.str.strip().str.upper()

lokacije = hd[
    ["UNITID", "INSTNM", "STABBR", "LATITUDE", "LONGITUD"]
].rename(columns={
    "INSTNM": "univerza",
    "LATITUDE": "latitude",
    "LONGITUD": "longitude"
})

#primerljive štiriletne javne in zasebne neprofitne institucije
inst = inst[
    inst["CONTROL"].isin([1, 2]) &
    (inst["ICLEVEL"] == 1)
][["UNITID", "leto"]].drop_duplicates()

#RV1 – stopnja sprejema
rv1["APPLCN"] = pd.to_numeric(rv1["APPLCN"], errors="coerce")
rv1["ADMSSN"] = pd.to_numeric(rv1["ADMSSN"], errors="coerce")

a = rv1[
    (rv1["APPLCN"] > 0) &
    (rv1["ADMSSN"] >= 0) &
    (rv1["ADMSSN"] <= rv1["APPLCN"])
][["UNITID", "leto", "APPLCN", "ADMSSN"]].copy()

a["stopnja_sprejema"] = 100 * a["ADMSSN"] / a["APPLCN"]

#RV2 – kohorta in zaključeni v 150 % predvidenega časa
rv2["GRTOTLT"] = pd.to_numeric(rv2["GRTOTLT"], errors="coerce")

b0 = rv2[
    (rv2["GRTYPE"] == 2) &
    (rv2["CHRTSTAT"] == 12) &
    (rv2["SECTION"] == 1) &
    (rv2["COHORT"] == 1)
][["UNITID", "leto", "GRTOTLT"]].rename(
    columns={"GRTOTLT": "kohorta"}
)

b1 = rv2[
    (rv2["GRTYPE"] == 3) &
    (rv2["CHRTSTAT"] == 13) &
    (rv2["SECTION"] == 1) &
    (rv2["COHORT"] == 1)
][["UNITID", "leto", "GRTOTLT"]].rename(
    columns={"GRTOTLT": "zakljucili"}
)

b = b0.merge(
    b1, on=["UNITID", "leto"],
    how="inner", validate="one_to_one"
)

b = b[
    (b["kohorta"] > 0) &
    (b["zakljucili"] >= 0) &
    (b["zakljucili"] <= b["kohorta"])
].copy()

b["stopnja_zakljucevanja"] = (
    100 * b["zakljucili"] / b["kohorta"]
)


#RV4 – diplome prve stopnje, glavna študijska smer
rv4["CIPCODE"] = pd.to_numeric(rv4["CIPCODE"], errors="coerce")
rv4["CRACE24"] = pd.to_numeric(rv4["CRACE24"], errors="coerce")
rv4["CTOTALT"] = pd.to_numeric(rv4["CTOTALT"], errors="coerce")

rv4["stevilo_diplom"] = rv4["CRACE24"].fillna(rv4["CTOTALT"])

c = rv4[
    (rv4["AWLEVEL"] == 5) &
    (rv4["MAJORNUM"] == 1)
].copy()

#skupno število diplom prve stopnje
c0 = c[
    np.isclose(c["CIPCODE"], 99)
][["UNITID", "leto", "stevilo_diplom"]].rename(
    columns={"stevilo_diplom": "diplome_skupaj"}
)

#podrobne kode CIP 11, brez agregata 11.0000
cip11 = c[
    (np.floor(c["CIPCODE"]) == 11) &
    (~np.isclose(c["CIPCODE"], 11))
].copy()

#seštevek diplom računalništva po univerzi in letu
c1 = (
    cip11.groupby(["UNITID", "leto"], as_index=False)
    ["stevilo_diplom"].sum(min_count=1)
    .rename(columns={"stevilo_diplom": "diplome_cip11"})
)

c = c0.merge(
    c1, on=["UNITID", "leto"],
    how="left", validate="one_to_one"
)

#nič diplom CIP 11, če univerza nima takega programa v podatkih
c["diplome_cip11"] = c["diplome_cip11"].fillna(0)

c = c[
    (c["diplome_skupaj"] > 0) &
    (c["diplome_cip11"] >= 0) &
    (c["diplome_cip11"] <= c["diplome_skupaj"])
].copy()

c["delez_cip11"] = (
    100 * c["diplome_cip11"] / c["diplome_skupaj"]
)

print("RV4 – pokritost po letih:")
display(c.groupby("leto")["UNITID"].nunique())


#združitev treh kazalnikov
podatki = inst.merge(
    a, on=["UNITID", "leto"], how="left"
).merge(
    b, on=["UNITID", "leto"], how="left"
).merge(
    c, on=["UNITID", "leto"], how="left"
).merge(
    lokacije, on="UNITID", how="left"
)

#veljavne koordinate za ameriški zemljevid
podatki["latitude"] = pd.to_numeric(podatki["latitude"], errors="coerce")
podatki["longitude"] = pd.to_numeric(podatki["longitude"], errors="coerce")

podatki = podatki[
    podatki["latitude"].between(18, 72) &
    podatki["longitude"].between(-180, -60)
].copy()

assert not podatki.duplicated(["UNITID", "leto"]).any()

#preverjanje pokritosti kazalnikov
print("Število institucij:", podatki["UNITID"].nunique())
print("Obdobje:", podatki["leto"].min(), "–", podatki["leto"].max())

pokritost = podatki.groupby("leto").agg(
    RV1=("stopnja_sprejema", "count"),
    RV2=("stopnja_zakljucevanja", "count"),
    RV4=("delez_cip11", "count")
)
display(pokritost)

#nastavitve kazalnikov
kazalniki = {
    "RV1": {
        "stolpec": "stopnja_sprejema",
        "naslov": "RV1 – Stopnja sprejema",
        "stevec": "ADMSSN",
        "imenovalec": "APPLCN",
        "opis_stevca": "Sprejeti",
        "opis_imenovalca": "Prijave"
    },
    "RV2": {
        "stolpec": "stopnja_zakljucevanja",
        "naslov": "RV2 – Stopnja zaključevanja",
        "stevec": "zakljucili",
        "imenovalec": "kohorta",
        "opis_stevca": "Zaključili",
        "opis_imenovalca": "Kohorta"
    },
    "RV4": {
        "stolpec": "delez_cip11",
        "naslov": "RV4 – Delež diplom CIP 11",
        "stevec": "diplome_cip11",
        "imenovalec": "diplome_skupaj",
        "opis_stevca": "Diplome CIP 11",
        "opis_imenovalca": "Vse diplome prve stopnje"
    }
}

#priprava podatkov za interaktivni zemljevid
zemljevid_podatki = {}

for leto in range(2004, 2025):
    zemljevid_podatki[str(leto)] = {}

    for rv, nastavitve in kazalniki.items():
        stolpec = nastavitve["stolpec"]

        df = podatki[
            (podatki["leto"] == leto) &
            podatki[stolpec].notna()
        ].copy()

        df = df[[
            "univerza", "STABBR", "latitude", "longitude",
            stolpec, nastavitve["stevec"], nastavitve["imenovalec"]
        ]]

        df.columns = [
            "univerza", "drzava", "latitude", "longitude",
            "vrednost", "stevec", "imenovalec"
        ]

        df = df.astype(object).where(pd.notna(df), None)
        zemljevid_podatki[str(leto)][rv] = df.to_dict("records")


#barvne meje za vsak RV na podlagi vseh razpoložljivih let
barvne_meje = {}

for rv, nastavitve in kazalniki.items():
    vrednosti = podatki[nastavitve["stolpec"]].dropna()

    barvne_meje[rv] = {
        "min": float(vrednosti.quantile(0.05)),
        "max": float(vrednosti.quantile(0.95))
    }

print("Barvne meje po kazalnikih:")
display(pd.DataFrame(barvne_meje).T.round(2))


#osnovni zemljevid
fig_zda = go.Figure(go.Scattergeo(
    lon=[], lat=[],
    mode="markers",
    hoverinfo="text",
    marker=dict(
        size=6,
        opacity=0.8,
        colorscale=[
            [0, "#b91c1c"],
            [0.25, "#f97316"],
            [0.5, "#facc15"],
            [0.75, "#86efac"],
            [1, "#15803d"]
        ],
        cmin=0,
        cmax=100,
        showscale=True,
        colorbar=dict(
            title=dict(text="Vrednost (%)"),
            thickness=16,
            len=0.7
        )
    )
))

fig_zda.update_layout(
    title=dict(
        text="Geografski vzorci ameriških univerz",
        x=0.5
    ),
    geo=dict(
        scope="usa",
        projection_type="albers usa",
        showland=True,
        landcolor="#F3F5F7",
        showlakes=True,
        lakecolor="white",
        showsubunits=True,
        subunitcolor="#CBD5E1",
        showcountries=True,
        countrycolor="#CBD5E1"
    ),
    height=690,
    margin=dict(l=20, r=25, t=75, b=20)
)

#pretvorba v HTML z dropdownom in drsnikom
id_grafa = "zemljevid-zda-rv"
html_graf = pio.to_html(
    fig_zda,
    full_html=False,
    include_plotlyjs=True,
    div_id=id_grafa
)

kontrolniki = """
<div style="font-family:Arial,sans-serif;max-width:1100px;">
  <div style="display:flex;gap:35px;flex-wrap:wrap;
              align-items:center;padding:12px;">
    <div>
      <label for="zda-rv"><b>Raziskovalni kazalnik</b></label><br>
      <select id="zda-rv" style="padding:8px;margin-top:6px;">
        <option value="RV1">RV1 – Stopnja sprejema</option>
        <option value="RV2">RV2 – Stopnja zaključevanja</option>
        <option value="RV4">RV4 – Delež diplom CIP 11</option>
      </select>
    </div>
    <div>
      <label for="zda-leto">
        <b>Leto: <span id="zda-leto-oznaka">2024</span></b>
      </label><br>
      <input id="zda-leto" type="range"
             min="2004" max="2024" step="1" value="2024"
             style="width:340px;margin-top:12px;">
    </div>
  </div>
  <p id="zda-opomba"
     style="margin:0 12px;color:#666;font-size:13px;"></p>
</div>
"""

javascript = """
<script>
(function () {
  const podatki = __PODATKI__;
  const kazalniki = __KAZALNIKI__;
  const barvneMeje = __BARVNE_MEJE__;

  const graf = document.getElementById("zemljevid-zda-rv");
  const izbiraRV = document.getElementById("zda-rv");
  const izbiraLeta = document.getElementById("zda-leto");
  const oznakaLeta = document.getElementById("zda-leto-oznaka");
  const opomba = document.getElementById("zda-opomba");

  function prikazi() {
    const rv = izbiraRV.value;
    const leto = izbiraLeta.value;
    const n = kazalniki[rv];
    const zapisi = podatki[leto][rv];

    oznakaLeta.textContent = leto;

    const hover = zapisi.map(z =>
      "<b>" + z.univerza + "</b><br>" +
      "Država: " + z.drzava + "<br>" +
      "Leto: " + leto + "<br>" +
      n.naslov + ": " + Number(z.vrednost).toFixed(2) + " %<br>" +
      n.opis_stevca + ": " +
      Number(z.stevec).toLocaleString("sl-SI") + "<br>" +
      n.opis_imenovalca + ": " +
      Number(z.imenovalec).toLocaleString("sl-SI")
    );

    Plotly.restyle(graf, {
      lon: [zapisi.map(z => z.longitude)],
      lat: [zapisi.map(z => z.latitude)],
      text: [hover],
      "marker.color": [zapisi.map(z => z.vrednost)],
      "marker.cmin": [barvneMeje[rv].min],
      "marker.cmax": [barvneMeje[rv].max],
      "marker.colorbar.title.text": [n.naslov + " (%)"]
    }, [0]);

    Plotly.relayout(graf, {
      "title.text": n.naslov + " – ameriške univerze (" + leto + ")"
    });

    opomba.textContent = zapisi.length
      ? "Prikazanih institucij: " + zapisi.length +
        ". Barvna lestvica 0–100 % je enaka skozi vsa leta."
      : "Za ta kazalnik in leto ni razpoložljivih podatkov.";
  }

  izbiraRV.addEventListener("change", prikazi);
  izbiraLeta.addEventListener("input", prikazi);

  prikazi();
})();
</script>
"""


javascript = (
    javascript
    .replace("__PODATKI__", json.dumps(
        zemljevid_podatki, ensure_ascii=False, allow_nan=False
    ))
    .replace("__KAZALNIKI__", json.dumps(
        kazalniki, ensure_ascii=False
    ))
    .replace("__BARVNE_MEJE__", json.dumps(barvne_meje))
)


celoten_html = (
    "<html><head><meta charset='utf-8'></head><body>" +
    kontrolniki + html_graf + javascript +
    "<p style='font:12px Arial;color:gray;margin:12px'>"
    "Vir: IPEDS/NCES, 2004–2024; koordinate HD2024. "
    "RV2: 2008–2024. Lastni izračuni.</p>" +
    "</body></html>"
)

#prikaz v Colabu
display(HTML(celoten_html))

#shranjevanje
pot_html = FIG_DIR / "RV1_RV2_RV4_zemljevid_ZDA_2004_2024.html"
pot_html.write_text(celoten_html, encoding="utf-8")

podatki.to_csv(
    DATA_DIR / "RV1_RV2_RV4_geografski_podatki.csv",
    index=False
)

print("Shranjeno:", pot_html)


RV4 – pokritost po letih:


,UNITID
leto,
2004,2200
2005,2271
2006,2290
2007,2338
2008,2380
2009,2426
2010,2485
2011,2537
2012,2616


Število institucij: 2538
Obdobje: 2004 – 2024


,RV1,RV2,RV4
leto,,,
2004,1474,0,1702
2005,1474,0,1717
2006,1478,0,1721
2007,1489,0,1736
2008,1497,1686,1751
2009,1516,1709,1773
2010,1510,1725,1787
2011,1522,1741,1800
2012,1536,1760,1835


Barvne meje po kazalnikih:


,min,max
RV1,27.37,97.44
RV2,18.85,87.78
RV4,0.00,9.87


Shranjeno: /content/figures/RV1_RV2_RV4_zemljevid_ZDA_2004_2024.html


RV1 - Odstopanja posameznih Ivy League univerz

In [59]:

#RV1 – odstopanja posameznih Ivy League univerz
rv1_ivy = pd.read_csv("/content/IPEDS_RV1.csv", low_memory=False)
rv1_ivy = rv1_ivy[rv1_ivy["UNITID"].isin(ivy_ids)].copy()
rv1_ivy["univerza"] = rv1_ivy["UNITID"].map(ivy_ids)

for stolpec in ["APPLCN", "ADMSSN", "ENRLT"]:
    rv1_ivy[stolpec] = pd.to_numeric(rv1_ivy[stolpec], errors="coerce")

#primerjava začetnega in končnega leta
primerjava = rv1_ivy[
    rv1_ivy["leto"].isin([2004, 2024])
].pivot(
    index="univerza",
    columns="leto",
    values=["APPLCN", "ADMSSN", "ENRLT"]
)

rezultati = pd.DataFrame(index=primerjava.index)

for stolpec, naziv in [
    ("APPLCN", "Prijave"),
    ("ADMSSN", "Sprejemi"),
    ("ENRLT", "Vpisi")
]:
    zacetek = primerjava[(stolpec, 2004)]
    konec = primerjava[(stolpec, 2024)]

    rezultati[naziv + " 2004"] = zacetek
    rezultati[naziv + " 2024"] = konec
    rezultati["Sprememba " + naziv + " (%)"] = (
        100 * (konec / zacetek - 1)
    ).where(zacetek > 0)

print("Primerjava Ivy League 2004–2024:")
display(rezultati.round(1))

#največje odstopanje pri prijavah glede na vpise
rezultati["Razlika rasti prijav in vpisov (ot)"] = (
    rezultati["Sprememba Prijave (%)"] -
    rezultati["Sprememba Vpisi (%)"]
)

print("Največje razlike med rastjo prijav in vpisov:")
display(
    rezultati[[
        "Sprememba Prijave (%)",
        "Sprememba Sprejemi (%)",
        "Sprememba Vpisi (%)",
        "Razlika rasti prijav in vpisov (ot)"
    ]].sort_values(
        "Razlika rasti prijav in vpisov (ot)",
        ascending=False
    ).round(1)
)


Primerjava Ivy League 2004–2024:


,Prijave 2004,Prijave 2024,Sprememba Prijave (%),Sprejemi 2004,Sprejemi 2024,Sprememba Sprejemi (%),Vpisi 2004,Vpisi 2024,Sprememba Vpisi (%)
univerza,,,,,,,,,
Brown,15285.0,48904.0,219.9,2505.0,2638.0,5.3,1429.0,1719.0,20.3
Columbia,17258.0,60115.0,248.3,2275.0,2400.0,5.5,1331.0,1558.0,17.1
Cornell,20822.0,62993.0,202.5,6130.0,5516.0,-10.0,3054.0,3525.0,15.4
Dartmouth,11734.0,31656.0,169.8,2173.0,1710.0,-21.3,1077.0,1182.0,9.7
Harvard,19690.0,54008.0,174.3,2054.0,1970.0,-4.1,1638.0,1647.0,0.5
Penn,18282.0,65226.0,256.8,3878.0,3523.0,-9.2,2433.0,2395.0,-1.6
Princeton,13695.0,40468.0,195.5,1733.0,1868.0,7.8,1175.0,1410.0,20.0
Yale,17735.0,57517.0,224.3,2014.0,2227.0,10.6,1353.0,1554.0,14.9


Največje razlike med rastjo prijav in vpisov:


,Sprememba Prijave (%),Sprememba Sprejemi (%),Sprememba Vpisi (%),Razlika rasti prijav in vpisov (ot)
univerza,,,,
Penn,256.8,-9.2,-1.6,258.3
Columbia,248.3,5.5,17.1,231.3
Yale,224.3,10.6,14.9,209.5
Brown,219.9,5.3,20.3,199.7
Cornell,202.5,-10.0,15.4,187.1
Princeton,195.5,7.8,20.0,175.5
Harvard,174.3,-4.1,0.5,173.7
Dartmouth,169.8,-21.3,9.7,160.0


Najbolj izstopa University of Pennsylvania (Penn), kjer se je število prijav povečalo za 256,8 %, število sprejetih pa zmanjšalo za 9,2 %. Število vpisanih se je zmanjšalo za 1,6 %. Stopnja sprejema se je tako znižala z 21,2 % na 5,4 %.

Najbolj izstopa University of Pennsylvania (Penn), kjer se je število prijav povečalo za 256,8 %, število sprejetih pa zmanjšalo za 9,2 %. Število vpisanih se je zmanjšalo za 1,6 %. Stopnja sprejema se je tako znižala z 21,2 % na 5,4 %.

Ugotovitev: Penn predstavlja izrazit primer povečevanja selektivnosti, saj je rast prijav močno presegla spremembe v številu sprejetih in vpisanih študentov.